# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Scope note — what this notebook can honestly claim

This notebook works from **`work/baseline_action_score.csv`**, the output of the Week 4 baseline
scoring rule (331,437 rows). That file contains only:

`rank, content_hash_id, baseline_score, action, reason_code, march_impressions, staleness_days`

It does **not** contain the wider feature set built in `w03_data_contract` from the FlyRank
Hugging Face warehouse (`avg_search_position`, `march_sessions`, `engagement_rate`, `march_clicks`,
or the `is_declining_proxy` future-window target). That means two things are honestly out of scope
here and are *not* faked below:

1. There is **no modeling target** in this file, so no classifier is trained and no ROC AUC is reported.
2. The classic "deliberate leakage trap" from `w03_data_contract` (train on `march_clicks`, which is
   part of the April-vs-March proxy definition) cannot be repeated here — `march_clicks` isn't a column
   in this file.

What **is** genuinely available and worth investigating in this file: `baseline_score`, `rank`,
`action`, and `reason_code` are not raw signals — they are *outputs of the Week 4 rule itself*, built
directly from `march_impressions` and `staleness_days`. That is a real, checkable leakage question:
if any of those four columns were ever fed into a future model as an "input feature," the model would
partly be learning from its own past decision. Sections 1–4 below build and test that honestly.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("work/baseline_action_score.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.head()


Rows: 331437
Columns: ['rank', 'content_hash_id', 'baseline_score', 'action', 'reason_code', 'march_impressions', 'staleness_days']


,rank,content_hash_id,baseline_score,action,reason_code,march_impressions,staleness_days
0,1,content_42ce26be1ec6be00,95.704131,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,4411,264.0
1,2,content_bea86ce3455100b0,94.513173,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,3670,232.0
2,3,content_097459d155cccb26,94.225199,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,37930,124.0
3,4,content_f2df5a8a9057783e,94.209147,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,35980,124.0
4,5,content_ac4e2d9d3bbb06de,94.184859,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,33348,124.0


In [2]:
# Raw signals available in this file (from the March 2026 development window):
#   - march_impressions : total observed Google Search impressions, March 2026
#   - staleness_days     : days since last content update, measured at end of March 2026
#
# Engineered features built from them for a feature vector:
#   - log_impressions        : log1p(march_impressions), to tame the heavy right skew
#   - staleness_missing      : 1 if staleness_days is missing, 0 otherwise (see section 2)
#   - staleness_days_filled  : staleness_days with missing values imputed at the median
#     of the OBSERVED values only (median computed on non-missing rows, never on the
#     full column, to avoid quietly leaking distributional info from the missing-flag itself)

feature_df = df[["content_hash_id", "march_impressions", "staleness_days"]].copy()

feature_df["log_impressions"] = np.log1p(feature_df["march_impressions"])
feature_df["staleness_missing"] = feature_df["staleness_days"].isna().astype(int)

observed_median = feature_df.loc[feature_df["staleness_missing"] == 0, "staleness_days"].median()
feature_df["staleness_days_filled"] = feature_df["staleness_days"].fillna(observed_median)

print("Median staleness_days among OBSERVED rows only:", observed_median)
feature_df.head()


Median staleness_days among OBSERVED rows only: 34.0


,content_hash_id,march_impressions,staleness_days,log_impressions,staleness_missing,staleness_days_filled
0,content_42ce26be1ec6be00,4411,264.0,8.392083,0,264.0
1,content_bea86ce3455100b0,3670,232.0,8.208219,0,232.0
2,content_097459d155cccb26,37930,124.0,10.543524,0,124.0
3,content_f2df5a8a9057783e,35980,124.0,10.490746,0,124.0
4,content_ac4e2d9d3bbb06de,33348,124.0,10.414783,0,124.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [3]:
notes = []

# march_impressions
notes.append({
    "feature": "march_impressions",
    "meaning": "Total observed Google Search impressions for the page during March 2026.",
    "missing_values": int(df["march_impressions"].isna().sum()),
    "missing_handling": "None missing in this file — used as-is; log1p() version added for skew.",
    "available_when": "After the March 2026 observation window closes (uses only March data).",
})

# staleness_days
n_missing = int(df["staleness_days"].isna().sum())
pct_missing = round(100 * n_missing / len(df), 1)
notes.append({
    "feature": "staleness_days",
    "meaning": "Days since the content was last updated, measured at end of March 2026.",
    "missing_values": n_missing,
    "missing_handling": (
        f"{pct_missing}% of rows are missing (no recorded last-update date). "
        "Filled with the median of OBSERVED rows only, plus a staleness_missing flag "
        "so the model/rule can tell 'genuinely fresh' apart from 'unknown update history'."
    ),
    "available_when": "Measured at end of March 2026 — available at the decision moment.",
})

# baseline_score / rank / action / reason_code — flagged as NOT usable raw features
for col, why in [
    ("baseline_score", "Deterministic function of march_impressions + staleness_days (the Week 4 rule's own output)."),
    ("rank", "Just the sort order of baseline_score — same leakage as baseline_score."),
    ("action", "Thresholded from baseline_score — a downstream decision, not an input signal."),
    ("reason_code", "Constant for every row in this file (see below) and derived from the same rule."),
]:
    notes.append({
        "feature": col,
        "meaning": "Output of the Week 4 baseline scoring rule, not an observed search/content signal.",
        "missing_values": int(df[col].isna().sum()),
        "missing_handling": "N/A — excluded as a feature (see Section 3/4).",
        "available_when": "Only available AFTER the rule has already run — never available before the decision it informs.",
    })

notes_df = pd.DataFrame(notes)
notes_df


,feature,meaning,missing_values,missing_handling,available_when
0,march_impressions,Total observed Google Search impressions for t...,0,None missing in this file — used as-is; log1p(...,After the March 2026 observation window closes...
1,staleness_days,"Days since the content was last updated, measu...",293358,88.5% of rows are missing (no recorded last-up...,Measured at end of March 2026 — available at t...
2,baseline_score,"Output of the Week 4 baseline scoring rule, no...",0,N/A — excluded as a feature (see Section 3/4).,Only available AFTER the rule has already run ...
3,rank,"Output of the Week 4 baseline scoring rule, no...",0,N/A — excluded as a feature (see Section 3/4).,Only available AFTER the rule has already run ...
4,action,"Output of the Week 4 baseline scoring rule, no...",0,N/A — excluded as a feature (see Section 3/4).,Only available AFTER the rule has already run ...
5,reason_code,"Output of the Week 4 baseline scoring rule, no...",0,N/A — excluded as a feature (see Section 3/4).,Only available AFTER the rule has already run ...


In [4]:
print("reason_code unique values:", df['reason_code'].unique())
print()
print(df['action'].value_counts())


reason_code unique values: <StringArray>
['STALE_HIGH_OPPORTUNITY']
Length: 1, dtype: str

action
MONITOR           321248
REVIEW_REFRESH     10189
Name: count, dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
from scipy.stats import spearmanr

# Attack: is baseline_score really just a (near-)deterministic function of the two raw
# signals, or does it carry independent information? If it's near-deterministic, using
# baseline_score/rank/action as an "input feature" anywhere downstream would just be
# re-feeding the rule's own output back into itself.

pearson_impr = df["march_impressions"].corr(df["baseline_score"])
spearman_impr = spearmanr(df["march_impressions"], df["baseline_score"]).correlation

staleness_filled = df["staleness_days"].fillna(df["staleness_days"].median())
pearson_stale = staleness_filled.corr(df["baseline_score"])
spearman_stale = spearmanr(staleness_filled, df["baseline_score"]).correlation

print("Pearson  march_impressions vs baseline_score:", round(pearson_impr, 4))
print("Spearman march_impressions vs baseline_score:", round(spearman_impr, 4))
print("Pearson  staleness_days(filled) vs baseline_score:", round(pearson_stale, 4))
print("Spearman staleness_days(filled) vs baseline_score:", round(spearman_stale, 4))


Pearson  march_impressions vs baseline_score: 0.3281
Spearman march_impressions vs baseline_score: 0.9646
Pearson  staleness_days(filled) vs baseline_score: 0.0795
Spearman staleness_days(filled) vs baseline_score: 0.09


In [6]:
# rank is by construction a monotonic transform of baseline_score - confirm directly
rank_check = (df.sort_values("baseline_score", ascending=False)["content_hash_id"].values
              == df.sort_values("rank")["content_hash_id"].values)
print("rank is a perfect re-sort of baseline_score:", rank_check.all())

# action is a threshold on baseline_score - find the cut point empirically
cut = df.loc[df["action"] == "REVIEW_REFRESH", "baseline_score"].min()
below_cut_all_monitor = (df.loc[df["baseline_score"] < cut, "action"] == "MONITOR").all()
print("Empirical action threshold on baseline_score:", round(cut, 4))
print("Every row below that threshold is MONITOR:", below_cut_all_monitor)

print()
print("LEAKAGE HUNT VERDICT:")
print("- march_impressions and staleness_days: independent raw signals, safe as features.")
print("- baseline_score: Spearman rank correlation ~0.96 with march_impressions alone;")
print("  it is a near-total re-encoding of the two raw signals, not new information.")
print("- rank: a strict re-sort of baseline_score (proven above) -> same leakage as baseline_score.")
print("- action: a hard threshold on baseline_score (proven above) -> same leakage, at coarser resolution.")
print("- reason_code: constant across all rows in this file -> carries zero information here.")


rank is a perfect re-sort of baseline_score: True
Empirical action threshold on baseline_score: 70.0
Every row below that threshold is MONITOR: True

LEAKAGE HUNT VERDICT:
- march_impressions and staleness_days: independent raw signals, safe as features.
- baseline_score: Spearman rank correlation ~0.96 with march_impressions alone;
  it is a near-total re-encoding of the two raw signals, not new information.
- rank: a strict re-sort of baseline_score (proven above) -> same leakage as baseline_score.
- action: a hard threshold on baseline_score (proven above) -> same leakage, at coarser resolution.
- reason_code: constant across all rows in this file -> carries zero information here.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:
excluded = pd.DataFrame([
    {"field": "baseline_score", "reason": "Output of the Week 4 rule itself; ~0.96 Spearman corr with march_impressions alone (Section 3). Using it as an input feature would feed the rule its own answer."},
    {"field": "rank",           "reason": "A strict re-sort of baseline_score (verified in Section 3) — identical leakage, different scale."},
    {"field": "action",         "reason": "A hard threshold applied to baseline_score (verified in Section 3) — same leakage, coarser resolution."},
    {"field": "reason_code",    "reason": "Constant for every row in this file (STALE_HIGH_OPPORTUNITY only) — zero discriminative information regardless of leakage status."},
])
excluded

kept = pd.DataFrame([
    {"field": "march_impressions",       "reason": "Raw observed March 2026 search signal; available before any rule/model runs."},
    {"field": "staleness_days",          "reason": "Raw observed content-age signal; available before any rule/model runs (with missingness handled explicitly, see Section 2)."},
    {"field": "log_impressions",         "reason": "Engineered from march_impressions only — no leakage introduced."},
    {"field": "staleness_missing",       "reason": "Engineered from staleness_days' own missingness pattern — no leakage introduced."},
]) 

print("EXCLUDED (rule-derived, leakage risk):")
display(excluded)
print()
print("KEPT (raw/engineered from raw only):")
display(kept)


EXCLUDED (rule-derived, leakage risk):


,field,reason
0,baseline_score,Output of the Week 4 rule itself; ~0.96 Spearm...
1,rank,A strict re-sort of baseline_score (verified i...
2,action,A hard threshold applied to baseline_score (ve...
3,reason_code,Constant for every row in this file (STALE_HIG...



KEPT (raw/engineered from raw only):


,field,reason
0,march_impressions,Raw observed March 2026 search signal; availab...
1,staleness_days,Raw observed content-age signal; available bef...
2,log_impressions,Engineered from march_impressions only — no le...
3,staleness_missing,Engineered from staleness_days' own missingnes...


**Honest limitation of this notebook:** because this file has no future-window target
(no `is_declining_proxy` / no April data), the leakage check above is about the feature
set being *internally self-referential* (rule outputs re-used as inputs), not about
target leakage against a real label. The target-leakage version of this exercise — the
`march_clicks` vs. `is_declining_proxy` trap — was already done honestly in
`w03_data_contract` (leaked ROC AUC 0.9677 → honest ROC AUC 0.9238) and isn't repeated
here because the columns it needs aren't in this file.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.